In [1]:
import sys
sys.path.insert(0, "..") # dalgebra is here

from dalgebra import *

# Looking for (particular) solutions to non-linear systems in integrable hierarchies

## 1. Introduction

One of the problems in the world of integrable hierarchies is to find linear differential operators with non-trivial centralizers, i.e., find elements $u_2,\ldots,u_n$ in some differential field $\mathbb{K}$
$$L = \partial^n + u_{2}\partial^{n-2} + ... + u_{n},$$
such that there is an operator $B \in \mathbb{K}[\partial] \setminus \mathbf{C}[L]$ such that $[L,B] = LB - BL = 0$.

In [[1]](https://doi.org/10.1007/s11786-025-00601-9), we show that we can work on this using _almost commuting differential operators_, i.e., operators $G \in \mathbb{K}[\partial]$ such that $\text{ord}([L,G]) \leq n-2$. We know that these almost commuting operators form a $\mathbf{C}$-vector space with a graded basis $(P_n)_{n\in \mathbb{N}}$ for which 
* $\text{ord}(P_n) = n$,
* $P_n$ is monic,
* coefficients of $P_n$ are _homogeneous_ w.r.t. a certain weight function (not important here).
Computing $[L,P_n]$ leads to an order $n-2$ linear operator. Since any operator that commutes is also almost commuting, then elements in the centralizer must be lienar combinations of the different $P_n$:
$$B = c_0 + c_1 P_1 + \ldots + c_m P_m,$$
and, more importantly, the value of $[L,B]$ is also $\mathbf{C}$-linear:
$$[L,B] = c_0 + c_1 [L,P_1] + \ldots + c_m [L,P_m].$$

In [[2]](https://foi.org/10.48550/arXiv.2505.01289), we exploited this property by stating the following result:

**Theorem** A fixed operator $L$ of order $n$ has an element in the centralizer of order $m$ if and only if the linear system over $\mathbb{K}$:
$$c_0 + c_1 [L,P_1] + \ldots + [L,P_m] = 0$$
has a constant solution.

In [2], we also studied what we called *level varieties*: we set up $L(\theta)$ with some constants parameters $\Theta = (\theta_1,\ldots,\theta_r)$, we define the level variety of $L$ at level $m$ to be the variety $\mathcal{M}_m \subset \mathbb{A}_{\mathbf{C}}^r$ such that
$$\theta \in \mathcal{M}_m \Leftrightarrow L(\theta)\text{ has a element commuting of order $\leq m$ not in $\mathbf{C}[L]$}.$$

If the ansatz is set up properly, we get algebraic varieties, i.e., defined by polynomial ideals.

**Goal of this notebook:** 
1. (Section 2) Present plain code to generate the algebraic systems (i.e., the level varieties) is a simple and generic way
2. (Section 3) Computing solutions "until we are happy" in the each corresponding level variety.
3. (Section 4) Study different examples arising from this as to state some questions for further work.

## 2. Script for getting the systems

#### Examples and code

We set up several examples for running the code and getting the level varieties:

In [2]:
# Trigonometric example (with sine and cosine)
B1.<a_2,a_3,a_4,a_5> = QQ[]
DB1 = DifferentialRing(B1); a_2,a_3,a_4,a_5 = DB1.gens()

E1.<cos> = DElliptic(DB1.fraction_field(), "cos^2 + cos_p^2 - 1")
sin = cos.derivative()

## Operators (tuples are the (u_m,...,u_2))
U14 = (a_4*(1/cos^2).derivative(times=2), a_3*(1/cos^2).derivative(), a_2*(1/cos^2))
U15 = (a_5*(1/cos^2).derivative(times=3), a_4*(1/cos^2).derivative(times=2), a_3*(1/cos^2).derivative(), a_2*(1/cos^2))

In [3]:
# P-Weierstrass example (with wp and wp_p)
B2.<b_2,b_3,b_4,b_5,g_2,g_3> = QQ[]
DB2 = DifferentialRing(B2); b_2,b_3,b_4,b_5,g_2,g_3 = DB2.gens()

E2.<wp> = DElliptic(DB2.fraction_field(), "wp_p^2 - 4*wp^3 - g_2*wp-g_3")
wp_p = wp.derivative()

## Operators (tuples are the (u_m,...,u_2))
U24 = (b_4*(wp).derivative(times=2), b_3*(wp).derivative(), b_2*(wp))
U25 = (b_5*(wp).derivative(times=3), b_4*(wp).derivative(times=2), b_3*(wp).derivative(), b_2*(wp))

Code for getting a level variety

In [4]:
from sage.structure.element import Element
from sage.rings.ideal import Ideal as ideal, Ideal_generic as Ideal
from dalgebra.commutators.commutator import GetEquationsForSolution

def level_variety(coeffs: tuple[Element], level: int, radical: bool = True) -> Ideal:
    L,P,I = GetEquationsForSolution(len(coeffs)+1, level, coeffs, analyze=False)

    return I.radical() if radical else I

#### Execution on examples

##### Trigonometric example from order 4 to level 5

In [5]:
I1_41 = level_variety(U14, 1)
I1_42 = level_variety(U14, 2)
I1_43 = level_variety(U14, 3)
I1_45 = level_variety(U14, 5)
I1_46 = level_variety(U14, 6, False)

In [6]:
I1_41

Ideal (a_4, a_3, a_2) of Multivariate Polynomial Ring in a_2, a_3, a_4, a_5 over Rational Field

In [7]:
I1_42

Ideal (a_4, a_3, a_2) of Multivariate Polynomial Ring in a_2, a_3, a_4, a_5 over Rational Field

In [8]:
I1_43

Ideal (a_2 - 2*a_3 + 4*a_4, a_4^2 + 4*a_4, a_3*a_4 + 12*a_4, a_3^2 + 4*a_3 + 24*a_4) of Multivariate Polynomial Ring in a_2, a_3, a_4, a_5 over Rational Field

In [9]:
I1_45

Ideal (a_2 - 2*a_3 + 4*a_4, a_3*a_4 - 3*a_4^2, a_4^3 + 16*a_4^2 + 48*a_4, a_3^3 + 16*a_3^2 + 288*a_4^2 + 48*a_3 + 1152*a_4) of Multivariate Polynomial Ring in a_2, a_3, a_4, a_5 over Rational Field

In [10]:
len(I1_46.gens()), max(el.degree() for el in I1_46.gens())

(7740, 12)

**Question:** can we compute a point in the variety of the last ideal that was not in the previous ideals?

##### Trigonometric example from order 5 to level 6

In [11]:
I1_51 = level_variety(U15, 1)
I1_52 = level_variety(U15, 2)
I1_53 = level_variety(U15, 3)
I1_54 = level_variety(U15, 4)
I1_56 = level_variety(U15, 6, False) # -> slow

In [12]:
I1_51

Ideal (a_5, a_4, a_3, a_2) of Multivariate Polynomial Ring in a_2, a_3, a_4, a_5 over Rational Field

In [13]:
I1_52

Ideal (a_5, a_4, a_3, a_2) of Multivariate Polynomial Ring in a_2, a_3, a_4, a_5 over Rational Field

In [14]:
I1_53

Ideal (a_5, a_4, a_3, a_2) of Multivariate Polynomial Ring in a_2, a_3, a_4, a_5 over Rational Field

In [15]:
I1_54

Ideal (a_5, a_4, a_3, a_2) of Multivariate Polynomial Ring in a_2, a_3, a_4, a_5 over Rational Field

In [16]:
len(I1_56.gens()), max(el.degree() for el in I1_56.gens())

(41760, 12)

**Question:** can we compute a point in the variety of the last ideal that was not in the previous ideals?

##### Elliptic example from order 4 to level 5

In [17]:
I2_41 = level_variety(U24, 1)
I2_42 = level_variety(U24, 2)
I2_43 = level_variety(U24, 3)
I2_45 = level_variety(U24, 5)
I2_46 = level_variety(U24, 6)

In [18]:
I2_41

Ideal (b_4, b_3, b_2) of Multivariate Polynomial Ring in b_2, b_3, b_4, b_5, g_2, g_3 over Rational Field

In [19]:
I2_42

Ideal (b_2 - b_3, b_3^2 + 12*b_3 - 24*b_4) of Multivariate Polynomial Ring in b_2, b_3, b_4, b_5, g_2, g_3 over Rational Field

In [20]:
I2_43

Ideal (b_2*b_3 - b_3^2 - 2*b_2*b_4 + 2*b_3*b_4 + 4*b_2 - 4*b_3, b_2^2 - b_3^2 - 2*b_2*b_4 + 2*b_3*b_4 + 12*b_2 - 12*b_3, b_3^2*g_2 + 4*b_2*b_4*g_2 - 4*b_3*b_4*g_2 - 8*b_2*g_2 + 20*b_3*g_2 - 24*b_4*g_2, b_2*b_4^2 - b_3*b_4^2 + 4*b_2*b_4 - 4*b_3*b_4, b_3^2*b_4 - 24*b_2*b_4 + 36*b_3*b_4 - 24*b_4^2, b_3^3 + 32*b_3^2 + 16*b_2*b_4 - 40*b_3*b_4 - 128*b_2 + 368*b_3 - 480*b_4) of Multivariate Polynomial Ring in b_2, b_3, b_4, b_5, g_2, g_3 over Rational Field

In [21]:
I2_45

Ideal (b_2^2*b_4 - 2*b_2*b_3*b_4 + b_3^2*b_4 + b_2*b_4^2 - b_3*b_4^2, 3*b_2^2*b_3 - 4*b_2*b_3^2 + b_3^3 - 2*b_2*b_3*b_4 + 2*b_3^2*b_4 + 2*b_2*b_4^2 - 2*b_3*b_4^2 + 20*b_2^2 + 20*b_2*b_3 - 40*b_3^2 - 80*b_2*b_4 + 80*b_3*b_4 + 240*b_2 - 240*b_3, 3*b_2^3 - 4*b_2*b_3^2 + b_3^3 - 2*b_2*b_3*b_4 + 2*b_3^2*b_4 + 2*b_2*b_4^2 - 2*b_3*b_4^2 + 104*b_2^2 - 40*b_2*b_3 - 64*b_3^2 - 128*b_2*b_4 + 128*b_3*b_4 + 816*b_2 - 816*b_3, 16*b_2*b_3^2*g_3 - 7*b_3^3*g_3 - 4*b_2*b_3*b_4*g_3 - 2*b_3^2*b_4*g_3 - 8*b_2*b_4^2*g_3 + 8*b_3*b_4^2*g_3 - 80*b_2^2*g_3 + 352*b_2*b_3*g_3 + 16*b_3^2*g_3 + 176*b_2*b_4*g_3 - 464*b_3*b_4*g_3 + 144*b_4^2*g_3 - 960*b_2*g_3 + 3120*b_3*g_3 - 4320*b_4*g_3, 4*b_3^3*g_2 + 24*b_2*b_3*b_4*g_2 - 27*b_3^2*b_4*g_2 - 29*b_2*b_4^2*g_2 + 29*b_3*b_4^2*g_2 - 68*b_2^2*g_2 + 136*b_2*b_3*g_2 + 64*b_3^2*g_2 + 188*b_2*b_4*g_2 - 320*b_3*b_4*g_2 + 72*b_4^2*g_2 - 816*b_2*g_2 + 1824*b_3*g_2 - 2016*b_4*g_2, 4*b_2*b_3^2*g_2 + 8*b_2*b_3*b_4*g_2 - 11*b_3^2*b_4*g_2 - 13*b_2*b_4^2*g_2 + 13*b_3*b_4^2*g_2 - 52*b

In [22]:
I2_46

Ideal (4*b_2^2*b_3*g_3 - b_3^3*g_3 - 4*b_2*b_3*b_4*g_3 + 2*b_3^2*b_4*g_3 + 144*b_2*b_3*g_3 - 48*b_3^2*g_3 - 48*b_2*b_4*g_3 - 48*b_3*b_4*g_3 + 48*b_4^2*g_3 + 720*b_3*g_3 - 1440*b_4*g_3, 3*b_2^3*g_2 - b_2^2*b_3*g_2 - 3*b_2*b_3^2*g_2 + b_3^3*g_2 + 96*b_2^2*g_2 - 48*b_2*b_3*g_2 - 48*b_3^2*g_2 - 96*b_2*b_4*g_2 + 96*b_3*b_4*g_2 + 720*b_2*g_2 - 720*b_3*g_2, b_2*b_3^3 - b_3^4 - 8*b_2^3*b_4 + 12*b_2^2*b_3*b_4 - 6*b_2*b_3^2*b_4 + 2*b_3^3*b_4 + 240*b_2^3 - 128*b_2^2*b_3 - 80*b_2*b_3^2 - 32*b_3^3 - 208*b_2^2*b_4 + 64*b_2*b_3*b_4 + 144*b_3^2*b_4 + 144*b_2*b_4^2 - 144*b_3*b_4^2 + 7680*b_2^2 - 3792*b_2*b_3 - 3888*b_3^2 - 7776*b_2*b_4 + 7776*b_3*b_4 + 57600*b_2 - 57600*b_3, b_2^2*b_3^2 - b_3^4 - 12*b_2^3*b_4 + 16*b_2^2*b_3*b_4 - 6*b_2*b_3^2*b_4 + 2*b_3^3*b_4 + 360*b_2^3 - 180*b_2^2*b_3 - 132*b_2*b_3^2 - 48*b_3^3 - 312*b_2^2*b_4 + 72*b_2*b_3*b_4 + 240*b_3^2*b_4 + 240*b_2*b_4^2 - 240*b_3*b_4^2 + 11520*b_2^2 - 5616*b_2*b_3 - 5904*b_3^2 - 11808*b_2*b_4 + 11808*b_3*b_4 + 86400*b_2 - 86400*b_3, b_2^3*b_3 - 

**Question:** can we compute a point in the variety of the last ideal that was not in the previous ideals?

##### Ellpitic example from order 5 to level 6

In [23]:
I2_51 = level_variety(U25, 1)
I2_52 = level_variety(U25, 2)
I2_53 = level_variety(U25, 3)
I2_54 = level_variety(U25, 4)
I2_56 = level_variety(U25, 6, False)

In [24]:
I2_51

Ideal (b_5, b_4, b_3, b_2) of Multivariate Polynomial Ring in b_2, b_3, b_4, b_5, g_2, g_3 over Rational Field

In [25]:
I2_52

Ideal (b_3 - 3*b_4 + 6*b_5, b_2 - 2*b_4 + 4*b_5, b_5*g_2, b_4*g_2, 4*b_5^2 + 5*b_5, b_4*b_5 + 5*b_5, 2*b_4^2 + 15*b_4 - 20*b_5) of Multivariate Polynomial Ring in b_2, b_3, b_4, b_5, g_2, g_3 over Rational Field

In [26]:
I2_53

Ideal (b_5*g_2, b_4*g_2, b_3*g_2, b_2*g_2, b_3*b_5 - 3*b_4*b_5 + 6*b_5^2, 10*b_2*b_5 - 15*b_4*b_5 + 36*b_5^2 + 20*b_5, 30*b_3*b_4 - 42*b_4^2 - 33*b_4*b_5 + 180*b_5^2 - 200*b_2 + 200*b_3 + 160*b_4 - 920*b_5, 15*b_2*b_4 - 6*b_4^2 - 24*b_4*b_5 + 72*b_5^2 - 50*b_2 + 50*b_3 + 130*b_4 - 470*b_5, 15*b_3^2 - 39*b_4^2 + 114*b_4*b_5 - 180*b_5^2 - 400*b_2 + 700*b_3 - 580*b_4 - 40*b_5, 5*b_2*b_3 - 6*b_4^2 + 21*b_4*b_5 - 36*b_5^2 - 50*b_2 + 150*b_3 - 170*b_4 + 70*b_5, 15*b_2^2 - 12*b_4^2 + 42*b_4*b_5 - 72*b_5^2 + 50*b_2 + 250*b_3 - 490*b_4 + 440*b_5, 24*b_5^3 + 125*b_4*b_5 - 390*b_5^2 + 100*b_5, 6*b_4*b_5^2 + 45*b_4*b_5 - 70*b_5^2 + 100*b_5, 3*b_4^2*b_5 + 70*b_4*b_5 - 60*b_5^2 + 200*b_5, 36*b_4^3 + 990*b_4^2 - 1245*b_4*b_5 + 900*b_5^2 - 2500*b_2 + 2500*b_3 + 2900*b_4 - 5500*b_5) of Multivariate Polynomial Ring in b_2, b_3, b_4, b_5, g_2, g_3 over Rational Field

In [27]:
I2_54

Ideal (b_5*g_2, b_4*g_2, b_3*g_2, b_2*g_2, b_2^2 + b_2*b_3 - 2*b_3^2 + b_2*b_4 + 2*b_3*b_4 - 8*b_2*b_5 + 30*b_2 - 30*b_3 + 30*b_4 - 60*b_5, 5984*b_3*b_5^2 - 9312*b_4*b_5^2 + 28992*b_5^3 + 47700*b_2*b_3 - 47700*b_3^2 - 35775*b_2*b_4 + 83475*b_3*b_4 - 35775*b_4^2 - 1130175*b_2*b_5 + 606930*b_3*b_5 - 293490*b_4*b_5 + 142560*b_5^2 + 357750*b_2 - 357750*b_3 + 357750*b_4 - 2789100*b_5, 2992*b_2*b_5^2 - 1824*b_4*b_5^2 + 8640*b_5^3 + 10500*b_2*b_3 - 10500*b_3^2 - 7875*b_2*b_4 + 18375*b_3*b_4 - 7875*b_4^2 - 524835*b_2*b_5 + 314250*b_3*b_5 - 189930*b_4*b_5 + 128480*b_5^2 + 78750*b_2 - 78750*b_3 + 78750*b_4 - 1155900*b_5, 110*b_4^2*b_5 - 228*b_4*b_5^2 + 112*b_5^3 + 7500*b_2*b_3 - 7500*b_3^2 - 5625*b_2*b_4 + 13125*b_3*b_4 - 5625*b_4^2 - 8975*b_2*b_5 - 12350*b_3*b_5 + 21015*b_4*b_5 - 20020*b_5^2 + 56250*b_2 - 56250*b_3 + 56250*b_4 - 115200*b_5, 7480*b_3*b_4*b_5 - 8832*b_4*b_5^2 + 28608*b_5^3 + 508500*b_2*b_3 - 508500*b_3^2 - 381375*b_2*b_4 + 889875*b_3*b_4 - 381375*b_4^2 - 152775*b_2*b_5 - 1290750*

In [28]:
len(I2_56.gens()), max(el.degree() for el in I2_56.gens())

(50944, 18)

**Question:** can we compute a point in the variety of the last ideal that was not in the previous ideals?

## 3. Open questions

To be filled